# Sample Data Analysis: Loading Data from S3

This notebook demonstrates how to securely load data from the S3 bucket provisioned by Terraform. The EC2 instance has an IAM role attached, which grants `boto3` the necessary permissions automatically without needing any hardcoded credentials.

### Step 1: Configure S3 Bucket and File Information

1. Run `terraform output s3_data_bucket_name` in your local terminal (from the `terraform/` directory) to get the name of the S3 bucket.
2. Paste the bucket name into the `bucket_name` variable in the cell below.
3. Upload the `sample_data.csv` file from the project's `data/` directory to your S3 bucket. You can use the AWS Console or the AWS CLI. With the CLI, run the following command from your project root, replacing `<YOUR_BUCKET_NAME_HERE>` with your bucket name:
   ```bash
   aws s3 cp data/sample_data.csv s3://<YOUR_BUCKET_NAME_HERE>/
   ```
4. Update the `file_key` variable if your filename is different.

In [ ]:
import boto3
import pandas as pd

# IMPORTANT: Replace with your S3 bucket name from `terraform output`
bucket_name = "<YOUR_BUCKET_NAME_HERE>"

# The name of the file in the S3 bucket
file_key = "sample_data.csv"

# The local path where the file will be downloaded
local_file_path = "/workspace/downloaded_data.csv"

### Step 2: Download the File from S3

The code below initializes the S3 client using `boto3`. Because an IAM role is attached to the EC2 instance, `boto3` automatically discovers and uses its permissions. No access keys are needed in the code.

In [ ]:
s3_client = boto3.client('s3')

try:
    s3_client.download_file(bucket_name, file_key, local_file_path)
    print(f"Successfully downloaded {file_key} from bucket {bucket_name} to {local_file_path}")
except Exception as e:
    print(f"Error downloading file: {e}")

### Step 3: Load and Display the Data

Now that the file is downloaded to the container's local filesystem, we can load it into a Pandas DataFrame.

In [ ]:
try:
    df = pd.read_csv(local_file_path)
    print("Data loaded successfully:")
    display(df.head())
except FileNotFoundError:
    print(f"File not found at {local_file_path}. Please ensure the download in the previous step was successful.")
except Exception as e:
    print(f"An error occurred while reading the CSV: {e}")